<a href="https://colab.research.google.com/github/ohNwghtEG/Multi-Asset-Portfolio-Performance-Risk-Dashboard/blob/main/Portfolio_Dash_Board.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ─────────────────────────────────────────
# CELL 1 — Install Libraries
# ─────────────────────────────────────────

In [ ]:
!pip install yfinance fredapi plotly pandas numpy quantstats -q

# ─────────────────────────────────────────
# CELL 2 — Set Up Your FRED API Key
# ─────────────────────────────────────────

In [ ]:
from google.colab import userdata
FRED_KEY = userdata.get('FRED_KEY')
print("FRED key loaded:", FRED_KEY[:6] + "..." if FRED_KEY else "NOT FOUND")

FRED key loaded: 990420...


# ─────────────────────────────────────────
# CELL 3 — Imports & Configuration
# ─────────────────────────────────────────

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from fredapi import Fred
import warnings
warnings.filterwarnings('ignore')

# ── Portfolio Configuration ──────────────────────────────────
# You can change these tickers and weights to anything you want.
# Weights must sum to 1.0.

TICKERS = ['SPY', 'AGG', 'GLD', 'QQQ', 'BTC-USD']
NAMES   = {
    'SPY':    'US Large Cap Equities',
    'AGG':    'US Aggregate Bonds',
    'GLD':    'Gold',
    'QQQ':    'Nasdaq 100 (Tech)',
    'BTC-USD':'Bitcoin',
}
WEIGHTS = {
    'SPY':    0.40,
    'AGG':    0.25,
    'GLD':    0.15,
    'QQQ':    0.10,
    'BTC-USD':0.10,
}
START_DATE = '2018-01-01'
END_DATE   = '2024-12-31'
BENCHMARK  = 'SPY'

print("Configuration loaded ✓")
print(f"Portfolio: {list(NAMES.values())}")

Configuration loaded ✓
Portfolio: ['US Large Cap Equities', 'US Aggregate Bonds', 'Gold', 'Nasdaq 100 (Tech)', 'Bitcoin']


# ─────────────────────────────────────────
# CELL 4 — Download Price Data
# ─────────────────────────────────────────


In [ ]:
print("Downloading price data from Yahoo Finance...")
raw = yf.download(TICKERS, start=START_DATE, end=END_DATE, auto_adjust=True, progress=False)['Close']
prices = raw.dropna()

print(f"✓ Downloaded {len(prices)} trading days × {len(prices.columns)} assets")
print(f"  Date range: {prices.index[0].date()} → {prices.index[-1].date()}")
print(f"\nFirst few rows:")
display(prices.head())

✓ Downloaded 1760 trading days × 5 assets
  Date range: 2018-01-02 → 2024-12-30

First few rows:


Ticker,AGG,BTC-USD,GLD,QQQ,SPY
Date,,,,,
2018-01-02,84.979607,14982.099609,125.150002,150.057251,235.954269
2018-01-03,84.987419,15201.000000,124.820000,151.515320,237.446762
2018-01-04,84.932953,15599.200195,125.459999,151.780380,238.447586
2018-01-05,84.878456,17429.500000,125.330002,153.304703,240.036469
2018-01-08,84.855087,15170.099609,125.309998,153.901199,240.475510


# ─────────────────────────────────────────
# CELL 5 — Compute Returns & Fetch Risk-Free Rate
# ─────────────────────────────────────────

In [ ]:
# Log returns (better statistical properties than simple returns)
log_returns = np.log(prices / prices.shift(1)).dropna()

# Simple returns for portfolio compounding
simple_returns = prices.pct_change().dropna()

# Pull 3-Month T-Bill rate from FRED as the risk-free rate
fred = Fred(api_key=FRED_KEY)
rf_series = fred.get_series('DGS3MO', observation_start=START_DATE) / 100  # convert % → decimal
rf_daily  = rf_series.reindex(log_returns.index, method='ffill') / 252     # annualize → daily
rf_daily  = rf_daily.fillna(rf_daily.median())

# Portfolio daily return (weighted sum of individual returns)
weights_array = np.array([WEIGHTS[t] for t in log_returns.columns])
portfolio_returns = log_returns.dot(weights_array)
portfolio_returns.name = 'Portfolio'

print("✓ Returns computed")
print(f"  Risk-free rate (latest): {rf_series.iloc[-1]:.2%}")

✓ Returns computed
  Risk-free rate (latest): 3.87%


# ─────────────────────────────────────────
# CELL 6 — Performance Metrics Function
# ─────────────────────────────────────────


In [ ]:
def compute_metrics(returns_df, rf_daily, label_map=None):
    '''
    Computes annualized risk/return metrics for each column in returns_df.
    Returns a styled DataFrame.
    '''
    results = {}
    for col in returns_df.columns:
        r  = returns_df[col].dropna()
        rf = rf_daily.reindex(r.index).ffill().fillna(0)

        ann_ret = r.mean() * 252
        ann_vol = r.std()  * np.sqrt(252)
        excess  = r - rf
        sharpe  = excess.mean() / r.std() * np.sqrt(252)

        downside_std = r[r < 0].std() * np.sqrt(252)
        sortino = ann_ret / downside_std if downside_std > 0 else np.nan

        cum = (1 + r).cumprod()
        rolling_max = cum.cummax()
        drawdown = (cum - rolling_max) / rolling_max
        max_dd = drawdown.min()

        calmar = ann_ret / abs(max_dd) if max_dd != 0 else np.nan

        # Value at Risk (95% confidence, historical)
        var_95 = np.percentile(r, 5)
        # Conditional VaR / Expected Shortfall
        cvar_95 = r[r <= var_95].mean()

        name = label_map.get(col, col) if label_map else col
        results[name] = {
            'Ann. Return':    f"{ann_ret:+.1%}",
            'Ann. Volatility':f"{ann_vol:.1%}",
            'Sharpe Ratio':   f"{sharpe:.2f}",
            'Sortino Ratio':  f"{sortino:.2f}",
            'Max Drawdown':   f"{max_dd:.1%}",
            'Calmar Ratio':   f"{calmar:.2f}",
            'Daily VaR (95%)':f"{var_95:.2%}",
            'CVaR (95%)':     f"{cvar_95:.2%}",
        }
    return pd.DataFrame(results).T


metrics = compute_metrics(log_returns, rf_daily, label_map=NAMES)
print("=" * 60)
print("PORTFOLIO METRICS SUMMARY")
print("=" * 60)
display(metrics)

PORTFOLIO METRICS SUMMARY


,Ann. Return,Ann. Volatility,Sharpe Ratio,Sortino Ratio,Max Drawdown,Calmar Ratio,Daily VaR (95%),CVaR (95%)
US Aggregate Bonds,+1.0%,6.0%,-0.24,0.20,-18.7%,0.05,-0.55%,-0.87%
Bitcoin,+26.1%,68.3%,0.35,0.48,-86.4%,0.30,-6.30%,-10.50%
Gold,+9.4%,14.3%,0.48,0.90,-23.9%,0.39,-1.49%,-2.12%
Nasdaq 100 (Tech),+17.6%,24.2%,0.63,0.92,-38.5%,0.46,-2.52%,-3.67%
US Large Cap Equities,+12.8%,19.5%,0.53,0.78,-35.7%,0.36,-1.87%,-3.06%


# ─────────────────────────────────────────
# CELL 7 — Chart 1: Cumulative Returns
# ─────────────────────────────────────────

In [ ]:
cum_returns = (1 + simple_returns).cumprod()

fig = go.Figure()

colors = ['#1f77b4', '#ff7f0e', '#ffd700', '#9467bd', '#d62728']

for i, ticker in enumerate(TICKERS):
    fig.add_trace(go.Scatter(
        x=cum_returns.index,
        y=cum_returns[ticker],
        name=NAMES[ticker],
        line=dict(width=2, color=colors[i]),
        hovertemplate=f'<b>{NAMES[ticker]}</b><br>Date: %{{x|%b %d, %Y}}<br>Growth of $1: $%{{y:.2f}}<extra></extra>'
    ))

fig.update_layout(
    title=dict(text='Cumulative Returns — Growth of $1 (2018–2024)', font=dict(size=18)),
    xaxis_title='Date',
    yaxis_title='Growth of $1',
    legend=dict(orientation='h', y=-0.15),
    hovermode='x unified',
    template='plotly_white',
    height=500,
)
fig.show()

# ─────────────────────────────────────────
# CELL 8 — Chart 2: Drawdown (Underwater) Chart
# ─────────────────────────────────────────

In [ ]:
fig = go.Figure()

for i, ticker in enumerate(TICKERS):
    cum = (1 + simple_returns[ticker]).cumprod()
    rolling_max = cum.cummax()
    dd = (cum - rolling_max) / rolling_max

    fig.add_trace(go.Scatter(
        x=dd.index,
        y=dd * 100,
        name=NAMES[ticker],
        fill='tozeroy',
        line=dict(width=1.5, color=colors[i]),
        fillcolor=colors[i].replace('rgb', 'rgba').replace(')', ', 0.15)') if 'rgb' in colors[i] else colors[i],
        hovertemplate=f'<b>{NAMES[ticker]}</b><br>Date: %{{x|%b %d, %Y}}<br>Drawdown: %{{y:.1f}}%<extra></extra>'
    ))

fig.update_layout(
    title=dict(text='Drawdown — Underwater Chart', font=dict(size=18)),
    xaxis_title='Date',
    yaxis_title='Drawdown (%)',
    yaxis=dict(ticksuffix='%'),
    legend=dict(orientation='h', y=-0.15),
    hovermode='x unified',
    template='plotly_white',
    height=450,
)
fig.show()


# ─────────────────────────────────────────
# CELL 9 — Chart 3: Rolling 60-Day Sharpe Ratio
# ────────────────────────────────────────

In [ ]:
fig = go.Figure()

for i, ticker in enumerate(TICKERS):
    rolling_sharpe = simple_returns[ticker].rolling(60).apply(
        lambda x: (x.mean() / x.std()) * np.sqrt(252) if x.std() > 0 else 0
    )
    fig.add_trace(go.Scatter(
        x=rolling_sharpe.index,
        y=rolling_sharpe,
        name=NAMES[ticker],
        line=dict(width=1.8, color=colors[i]),
    ))

# Zero line
fig.add_hline(y=0, line_dash='dash', line_color='red', line_width=1,
              annotation_text='Break-even', annotation_position='bottom right')
fig.add_hline(y=1, line_dash='dot', line_color='green', line_width=1,
              annotation_text='Sharpe = 1', annotation_position='top right')

fig.update_layout(
    title=dict(text='Rolling 60-Day Annualized Sharpe Ratio', font=dict(size=18)),
    xaxis_title='Date',
    yaxis_title='Sharpe Ratio (Annualized)',
    legend=dict(orientation='h', y=-0.15),
    hovermode='x unified',
    template='plotly_white',
    height=450,
)
fig.show()

# ─────────────────────────────────────────
# CELL 10 — Chart 4: Correlation Heatmap
# ─────────────────────────────────────────

In [ ]:
corr_matrix = simple_returns.corr()
corr_matrix.index   = [NAMES[t] for t in corr_matrix.index]
corr_matrix.columns = [NAMES[t] for t in corr_matrix.columns]

fig = px.imshow(
    corr_matrix,
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    text_auto='.2f',
    title='Asset Return Correlation Matrix (Full Period)',
    aspect='auto',
)
fig.update_layout(
    template='plotly_white',
    height=450,
    coloraxis_colorbar=dict(title='Correlation'),
)
fig.show()

print("\nKey insight: Look for low/negative correlations (blue) —")
print("these are the diversification pairs that reduce portfolio risk.")


Key insight: Look for low/negative correlations (blue) —
these are the diversification pairs that reduce portfolio risk.



# ─────────────────────────────────────────
# CELL 11 — Chart 5: Return Distribution with VaR Lines
# ─────────────────────────────────────────

In [ ]:
fig = go.Figure()

for i, ticker in enumerate(TICKERS):
    r = simple_returns[ticker].dropna()
    var_95  = np.percentile(r, 5)
    cvar_95 = r[r <= var_95].mean()

    fig.add_trace(go.Histogram(
        x=r * 100,
        name=NAMES[ticker],
        opacity=0.5,
        nbinsx=80,
        marker_color=colors[i],
    ))

fig.update_layout(
    title=dict(text='Daily Return Distributions', font=dict(size=18)),
    xaxis_title='Daily Return (%)',
    yaxis_title='Frequency',
    barmode='overlay',
    legend=dict(orientation='h', y=-0.15),
    template='plotly_white',
    height=450,
)
fig.show()

# ─────────────────────────────────────────
# CELL 12 — Chart 6: Bull/Bear Regime with 200-Day MA
# ─────────────────────────────────────────

In [ ]:
spy_prices = prices['SPY']
ma_200 = spy_prices.rolling(200).mean()

# Classify regime
bull_mask = spy_prices > ma_200

fig = go.Figure()

# Background shading for bear regimes
in_bear = False
bear_start = None
for date, is_bull in bull_mask.items():
    if not is_bull and not in_bear:
        bear_start = date
        in_bear = True
    elif is_bull and in_bear:
        fig.add_vrect(x0=bear_start, x1=date,
                      fillcolor='red', opacity=0.08, line_width=0)
        in_bear = False

fig.add_trace(go.Scatter(x=spy_prices.index, y=spy_prices,
                          name='SPY Price', line=dict(color='black', width=1.5)))
fig.add_trace(go.Scatter(x=ma_200.index, y=ma_200,
                          name='200-Day MA', line=dict(color='orange', width=2, dash='dash')))

fig.update_layout(
    title=dict(text='SPY Price with 200-Day MA Bull/Bear Regime (Red = Bear)', font=dict(size=18)),
    xaxis_title='Date',
    yaxis_title='Price (USD)',
    legend=dict(orientation='h', y=-0.15),
    template='plotly_white',
    height=450,
)
fig.show()

# ─────────────────────────────────────────
# CELL 13 — Portfolio-Level Summary Dashboard (Combined)
# ─────────────────────────────────────────

In [ ]:
# Compute portfolio cumulative returns
portfolio_cum = (1 + portfolio_returns).cumprod()
spy_cum       = (1 + simple_returns['SPY']).cumprod()

# Portfolio drawdown
rolling_max_port = portfolio_cum.cummax()
port_dd = (portfolio_cum - rolling_max_port) / rolling_max_port

fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    subplot_titles=[
        'Portfolio vs SPY Benchmark — Cumulative Return',
        'Portfolio Drawdown',
        'Rolling 60-Day Portfolio Sharpe',
    ],
    vertical_spacing=0.07,
    row_heights=[0.5, 0.25, 0.25],
)

fig.add_trace(go.Scatter(x=portfolio_cum.index, y=portfolio_cum,
    name='Portfolio', line=dict(color='steelblue', width=2.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=spy_cum.index, y=spy_cum,
    name='SPY Benchmark', line=dict(color='gray', width=1.5, dash='dash')), row=1, col=1)

fig.add_trace(go.Scatter(x=port_dd.index, y=port_dd * 100,
    name='Drawdown', fill='tozeroy', line=dict(color='red', width=1),
    fillcolor='rgba(255,0,0,0.15)'), row=2, col=1)

rolling_sharpe_port = portfolio_returns.rolling(60).apply(
    lambda x: (x.mean() / x.std()) * np.sqrt(252) if x.std() > 0 else 0
)
fig.add_trace(go.Scatter(x=rolling_sharpe_port.index, y=rolling_sharpe_port,
    name='Rolling Sharpe', line=dict(color='green', width=1.5)), row=3, col=1)
fig.add_hline(y=0, line_dash='dash', line_color='red', row=3, col=1)

fig.update_layout(
    title=dict(text='Portfolio Dashboard (40% SPY / 25% AGG / 15% GLD / 10% QQQ / 10% BTC)', font=dict(size=16)),
    height=750,
    template='plotly_white',
    showlegend=True,
    legend=dict(orientation='h', y=-0.05),
)
fig.update_yaxes(title_text='Growth of $1', row=1, col=1)
fig.update_yaxes(title_text='Drawdown (%)', ticksuffix='%', row=2, col=1)
fig.update_yaxes(title_text='Sharpe Ratio', row=3, col=1)
fig.show()

# Print final summary stats
port_ann_ret = portfolio_returns.mean() * 252
port_ann_vol = portfolio_returns.std()  * np.sqrt(252)
port_sharpe  = port_ann_ret / port_ann_vol
port_max_dd  = port_dd.min()
print("\n" + "=" * 50)
print("PORTFOLIO SUMMARY (2018–2024)")
print("=" * 50)
print(f"  Annualized Return : {port_ann_ret:+.1%}")
print(f"  Annualized Vol    : {port_ann_vol:.1%}")
print(f"  Sharpe Ratio      : {port_sharpe:.2f}")
print(f"  Max Drawdown      : {port_max_dd:.1%}")
print(f"  Final Value ($1)  : ${portfolio_cum.iloc[-1]:.2f}")


PORTFOLIO SUMMARY (2018–2024)
  Annualized Return : +11.2%
  Annualized Vol    : 14.6%
  Sharpe Ratio      : 0.76
  Max Drawdown      : -29.2%
  Final Value ($1)  : $2.02
